# Feature engineering::

#### Import and setup

In [ ]:
import pandas as pd
import numpy as np
import yaml
import os

os.chdir("..")

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("Ready.")

#### Load & filter raw data

In [ ]:
filepath = config["data"]["raw_dir"] + config["data"]["lending_club_file"]
df = pd.read_csv(filepath, low_memory=False)

# Keep only completed loans
df = df[df["loan_status"].isin(["Fully Paid", "Charged Off"])].copy()

# Binary target
df["default"] = (df["loan_status"] == "Charged Off").astype(int)

print(f"Rows: {len(df):,}")

#### Select relevant columns

In [ ]:
features = [
    "loan_amnt", "int_rate", "installment",
    "annual_inc", "dti", "emp_length",
    "fico_range_low", "fico_range_high",
    "revol_util", "revol_bal",
    "open_acc", "total_acc", "pub_rec",
    "delinq_2yrs", "inq_last_6mths",
    "home_ownership", "purpose", "grade", "term"
]

target = "default"

df = df[features + [target]].copy()
print(f"Shape: {df.shape}")
df.head()

#### Clean numeric columns

In [ ]:
# Strip % signs and convert to float
df["int_rate"] = df["int_rate"].str.replace("%", "").astype(float)
df["revol_util"] = df["revol_util"].str.replace("%", "").astype(float)

# Convert term to integer months
df["term"] = df["term"].str.strip().str.replace(" months", "").astype(int)

# Convert emp_length to numeric
emp_map = {
    "< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3,
    "4 years": 4, "5 years": 5, "6 years": 6, "7 years": 7,
    "8 years": 8, "9 years": 9, "10+ years": 10
}
df["emp_length"] = df["emp_length"].map(emp_map)

print("Numeric cleaning done.")
df[["int_rate", "revol_util", "term", "emp_length"]].head()

#### Nulls

In [ ]:
# Numeric nulls — fill with median
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_cols.remove("default")

for col in numeric_cols:
    median = df[col].median()
    null_count = df[col].isnull().sum()
    if null_count > 0:
        print(f"{col}: filling {null_count:,} nulls with median {median:.2f}")
    df[col] = df[col].fillna(median)

# Categorical nulls — fill with "Unknown"
cat_cols = df.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    df[col] = df[col].fillna("Unknown")

print("\nNull counts after cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("All nulls handled." if df.isnull().sum().sum() == 0 else "Still has nulls.")

#### Feature engineering

In [ ]:
# FICO midpoint
df["fico_score"] = (df["fico_range_low"] + df["fico_range_high"]) / 2

# Loan to income ratio
df["loan_to_income"] = df["loan_amnt"] / (df["annual_inc"] + 1)

# Monthly debt burden
df["monthly_debt"] = df["installment"] / (df["annual_inc"] / 12 + 1)

# Credit utilization flag
df["high_utilization"] = (df["revol_util"] > 75).astype(int)

# Derogatory mark flag
df["has_derog"] = (df["pub_rec"] > 0).astype(int)

print("Engineered features added.")
df[["fico_score", "loan_to_income", "monthly_debt", 
    "high_utilization", "has_derog"]].describe()

#### Encode categoricals

In [ ]:
# One-hot encode low-cardinality categoricals
df = pd.get_dummies(df, columns=["home_ownership", "purpose", "grade"], 
                    drop_first=True)

# Drop original term (already numeric)
print(f"Shape after encoding: {df.shape}")
df.head()

##### Save data:

In [ ]:
output_path = config["data"]["features_dir"] + "features.parquet"
df.to_parquet(output_path, index=False)
print(f"Saved {len(df):,} rows to {output_path}")
print(f"Final feature count: {df.shape[1] - 1}")  # minus target